In [ ]:
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification
import torch

# Load the CSV file
df = pd.read_csv('/content/for bert.csv')

# Drop rows with missing values in 'tweet_cleaned'
df.dropna(subset=['tweet_cleaned'], inplace=True)

# Extract tweets and labels
tweets = df['tweet_cleaned'].tolist()
labels = df['sentiment'].tolist()

# Create a mapping from sentiment labels to numerical values
sentiment_mapping = {
    'Neutral': 0,
    'Negative': 1,
    'Positive': 2  # Add more labels as needed
}

# Convert string labels to numerical labels
numerical_labels = [sentiment_mapping[label] for label in labels]

# Tokenize the tweets
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
input_ids = []
attention_mask = []
for tweet in tweets:
    encoded_dict = tokenizer.encode_plus(
        text=tweet,
        add_special_tokens=True,
        max_length=128,  # Adjust the maximum length as needed
        truncation=True,
        padding='max_length',
        return_attention_mask=True,
        return_tensors='pt'
    )
    input_ids.append(encoded_dict['input_ids'])
    attention_mask.append(encoded_dict['attention_mask'])

# Convert lists to tensors
input_ids = torch.cat(input_ids, dim=0) # Concatenate instead of stack
attention_mask = torch.cat(attention_mask, dim=0) # Concatenate instead of stack
# Use numerical_labels instead of labels
labels = torch.tensor(numerical_labels)

In [ ]:
# Load the pre-trained BERT model
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)  # Assuming 3 sentiment classes

# Fine-tune the model
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
loss_fn = torch.nn.CrossEntropyLoss()

# Define the number of training epochs and batch size
epochs = 10  # Set the desired number of epochs
batch_size = 32  # Set the desired batch size

model.train()
for epoch in range(epochs):
    for batch in range(len(input_ids) // batch_size):
        batch_input_ids = input_ids[batch * batch_size:(batch + 1) * batch_size]
        batch_attention_mask = attention_mask[batch * batch_size:(batch + 1) * batch_size]
        batch_labels = labels[batch * batch_size:(batch + 1) * batch_size]

        outputs = model(batch_input_ids, attention_mask=batch_attention_mask)
        loss = loss_fn(outputs.logits, batch_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

# Evaluate the model
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for batch in range(len(input_ids) // batch_size):
        batch_input_ids = input_ids[batch * batch_size:(batch + 1) * batch_size]
        batch_attention_mask = attention_mask[batch * batch_size:(batch + 1) * batch_size]
        batch_labels = labels[batch * batch_size:(batch + 1) * batch_size]

        outputs = model(batch_input_ids, attention_mask=batch_attention_mask)
        predicted_labels = torch.argmax(outputs.logits, dim=1)
        total += batch_labels.size(0)
        correct += (predicted_labels == batch_labels).sum().item()

    accuracy = correct / total
    print('Accuracy:', accuracy)

In [ ]:
# prompt: write codes to save the model so it can be deployed

model.save_pretrained('/content/fine_tuned_bert_model')
tokenizer.save_pretrained('/content/fine_tuned_bert_model')

# You can also save the model's state dictionary if needed.
# torch.save(model.state_dict(), '/content/fine_tuned_bert_model/model_state_dict.pth')

print("Model saved to /content/fine_tuned_bert_model")


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Evaluate the model
model.eval()
all_predicted_labels = []
all_true_labels = []
total_loss = 0

with torch.no_grad():
    # Iterate over batches for evaluation
    for i in range(0, len(input_ids), batch_size): # Use the previously defined 'batch_size'
        batch_input_ids = input_ids[i:i + batch_size]
        batch_attention_mask = attention_mask[i:i + batch_size]
        batch_labels = labels[i:i + batch_size]

        outputs = model(batch_input_ids, attention_mask=batch_attention_mask, labels=batch_labels)
        loss = outputs.loss
        total_loss += loss.item()

        predicted_labels = torch.argmax(outputs.logits, dim=1).tolist()
        all_predicted_labels.extend(predicted_labels)
        all_true_labels.extend(batch_labels.tolist())

# Calculate average loss
average_loss = total_loss / (len(input_ids) / batch_size)


# Calculate evaluation metrics
accuracy = accuracy_score(all_true_labels, all_predicted_labels)
f1 = f1_score(all_true_labels, all_predicted_labels, average='weighted')
recall = recall_score(all_true_labels, all_predicted_labels, average='weighted')
precision = precision_score(all_true_labels, all_predicted_labels, average='weighted')

# Calculate confusion matrix
cm = confusion_matrix(all_true_labels, all_predicted_labels)

# Print the metrics
print('Average Loss:', average_loss)
print('Accuracy:', accuracy)
print('F1-score:', f1)
print('Recall:', recall)
print('Precision:', precision)

# Plot the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=sentiment_mapping.keys(), yticklabels=sentiment_mapping.keys())
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
!pip install streamlit

Now that you have trained and saved your model, you can create a simple interface to test it with new text data. We will use Streamlit for this.

In [ ]:
import streamlit as st
import torch
from transformers import BertTokenizer, BertForSequenceClassification

# Load the fine-tuned model and tokenizer
# Make sure the path matches where you saved your model
model_path = '/content/fine_tuned_bert_model'
model = BertForSequenceClassification.from_pretrained(model_path)
tokenizer = BertTokenizer.from_pretrained(model_path)

# Define the sentiment mapping
sentiment_mapping = {
    0: 'Neutral',
    1: 'Negative',
    2: 'Positive'
}

# Set the model to evaluation mode
model.eval()

def predict_sentiment(text):
    """Predicts the sentiment of a given text."""
    # Tokenize the input text
    encoded_dict = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=128,
        truncation=True,
        padding='max_length',
        return_attention_mask=True,
        return_tensors='pt'
    )

    input_ids = encoded_dict['input_ids']
    attention_mask = encoded_dict['attention_mask']

    # Make prediction
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        predicted_class_id = torch.argmax(logits, dim=1).item()

    return sentiment_mapping[predicted_class_id]

# Create the Streamlit interface
st.title("Sentiment Prediction with BERT")

st.write("Enter text below to get the sentiment prediction.")

user_input = st.text_area("Enter text here:")

if st.button("Predict"):
    if user_input:
        sentiment = predict_sentiment(user_input)
        st.write(f"Predicted Sentiment: **{sentiment}**")
    else:
        st.write("Please enter some text to predict.")

In [ ]:
!pip install --upgrade transformers

To run this Streamlit app, save the code above as a Python file (e.g., `app.py`) and run the following command in your Colab terminal: